In [24]:
import pygame
import sys
import random
import copy

# --- Sudoku Logic ---
def find_next_empty(puzzle):
    for r in range(9):
        for c in range(9):
            if puzzle[r][c] == -1:
                return r, c
    return None, None

def is_valid(puzzle, guess, row, col):
    if guess in puzzle[row]:
        return False
    if guess in [puzzle[i][col] for i in range(9)]:
        return False
    box_row, box_col = 3 * (row // 3), 3 * (col // 3)
    for r in range(box_row, box_row + 3):
        for c in range(box_col, box_col + 3):
            if puzzle[r][c] == guess:
                return False
    return True

def solve_sudoku(puzzle):
    row, col = find_next_empty(puzzle)
    if row is None:
        return True
    for guess in range(1, 10):
        if is_valid(puzzle, guess, row, col):
            puzzle[row][col] = guess
            if solve_sudoku(puzzle):
                return True
            puzzle[row][col] = -1
    return False

# --- Pygame Setup ---
pygame.init()
WIDTH, HEIGHT = 540, 600
GRID_SIZE = 9
CELL_SIZE = WIDTH // GRID_SIZE

WHITE = (255, 255, 255)
BLACK = (0, 0, 0)
BLUE = (50, 50, 255)
LIGHT_BLUE = (173, 216, 230)
GRAY = (200, 200, 200)
GREEN = (0, 180, 0)
RED = (255, 0, 0)

FONT = pygame.font.SysFont("comicsans", 40)
BUTTON_FONT = pygame.font.SysFont("comicsans", 24)
SMALL_FONT = pygame.font.SysFont("comicsans", 20)

selected = None
conflict_cell = None
history = []

def generate_full_sudoku():
    grid = [[-1 for _ in range(9)] for _ in range(9)]
    solve_sudoku(grid)
    return grid

def generate_puzzle(full_grid, num_holes=40):
    puzzle = [row[:] for row in full_grid]
    count = 0
    while count < num_holes:
        row = random.randint(0, 8)
        col = random.randint(0, 8)
        if puzzle[row][col] != -1:
            puzzle[row][col] = -1
            count += 1
    return puzzle

def is_valid_move(grid, row, col, num):
    for i in range(9):
        if grid[row][i] == num or grid[i][col] == num:
            return False
    box_row, box_col = 3 * (row // 3), 3 * (col // 3)
    for r in range(3):
        for c in range(3):
            if grid[box_row + r][box_col + c] == num:
                return False
    return True

def is_puzzle_complete(grid):
    for r in range(9):
        for c in range(9):
            num = grid[r][c]
            if num == -1 or not is_valid_move(grid, r, c, num):
                return False
    return True

def draw_grid(win, grid, original_grid, conflict_cell):
    win.fill(WHITE)
    if selected:
        r, c = selected
        pygame.draw.rect(win, LIGHT_BLUE, (c * CELL_SIZE, r * CELL_SIZE, CELL_SIZE, CELL_SIZE))

    for r in range(GRID_SIZE):
        for c in range(GRID_SIZE):
            if grid[r][c] != -1:
                value = str(grid[r][c])
                if conflict_cell and (r, c) == conflict_cell:
                    color = RED
                else:
                    color = BLACK if original_grid[r][c] != -1 else BLUE
                text = FONT.render(value, True, color)
                text_rect = text.get_rect(center=(c * CELL_SIZE + CELL_SIZE // 2, r * CELL_SIZE + CELL_SIZE // 2))
                win.blit(text, text_rect)

    for i in range(GRID_SIZE + 1):
        thickness = 3 if i % 3 == 0 else 1
        pygame.draw.line(win, BLACK, (0, i * CELL_SIZE), (WIDTH, i * CELL_SIZE), thickness)
        pygame.draw.line(win, BLACK, (i * CELL_SIZE, 0), (i * CELL_SIZE, WIDTH), thickness)

def draw_buttons(win):
    clear_btn = pygame.Rect(30, 542, 100, 30)
    solve_btn = pygame.Rect(WIDTH - 130, 542, 100, 30)
    new_btn = pygame.Rect(WIDTH // 2 - 50, 542, 100, 30)

    pygame.draw.rect(win, GRAY, clear_btn)
    pygame.draw.rect(win, GRAY, solve_btn)
    pygame.draw.rect(win, GRAY, new_btn)

    # Center button labels
    for btn, label in zip([clear_btn, solve_btn, new_btn], ["Clear", "Solve", "New"]):
        text = BUTTON_FONT.render(label, True, BLACK)
        text_rect = text.get_rect(center=btn.center)
        win.blit(text, text_rect)

    return clear_btn, solve_btn, new_btn

def draw_message(win, message):
    if message:
        color = RED if "Invalid" in message else GREEN
        text = SMALL_FONT.render(message, True, color)
        text_rect = text.get_rect(center=(WIDTH // 2, HEIGHT - 22))  
        win.blit(text, text_rect)

def get_clicked_cell(pos):
    x, y = pos
    if y >= WIDTH:
        return None
    return y // CELL_SIZE, x // CELL_SIZE

def main():
    global selected, conflict_cell, history
    win = pygame.display.set_mode((WIDTH, HEIGHT))
    pygame.display.set_caption("Sudoku Game")

    full_solution = generate_full_sudoku()
    grid = generate_puzzle(full_solution, num_holes=40)
    original_grid = [row[:] for row in grid]

    solved_message = ""
    running = True

    while running:
        draw_grid(win, grid, original_grid, conflict_cell)
        clear_btn, solve_btn, new_btn = draw_buttons(win)
        draw_message(win, solved_message)
        pygame.display.update()

        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                running = False
                pygame.quit()
                return

            elif event.type == pygame.MOUSEBUTTONDOWN:
                pos = pygame.mouse.get_pos()
                selected = get_clicked_cell(pos)
                if clear_btn.collidepoint(pos):
                    grid = [row[:] for row in original_grid]
                    solved_message = ""
                    conflict_cell = None
                    history.clear()
                elif solve_btn.collidepoint(pos):
                    temp = [row[:] for row in grid]
                    if solve_sudoku(temp):
                        grid = temp
                        solved_message = "Puzzle Solved!"
                        conflict_cell = None
                    else:
                        solved_message = "Unsolvable Puzzle!"
                elif new_btn.collidepoint(pos):
                    full_solution = generate_full_sudoku()
                    grid = generate_puzzle(full_solution, num_holes=40)
                    original_grid = [row[:] for row in grid]
                    solved_message = ""
                    conflict_cell = None
                    history.clear()

            elif event.type == pygame.KEYDOWN and selected:
                r, c = selected
                if original_grid[r][c] == -1:
                    if event.key in (pygame.K_DELETE, pygame.K_BACKSPACE, pygame.K_0):
                        history.append((r, c, grid[r][c]))
                        grid[r][c] = -1
                        conflict_cell = None
                        solved_message = ""
                    elif pygame.K_1 <= event.key <= pygame.K_9:
                        num = event.key - pygame.K_0
                        if is_valid_move(grid, r, c, num):
                            history.append((r, c, grid[r][c]))
                            grid[r][c] = num
                            conflict_cell = None
                            solved_message = ""
                            if is_puzzle_complete(grid):
                                solved_message = "Puzzle Solved!"
                        else:
                            conflict_cell = (r, c)
                            solved_message = "Invalid move!"
                    elif event.key == pygame.K_z and history:
                        prev_r, prev_c, prev_val = history.pop()
                        grid[prev_r][prev_c] = prev_val
                        conflict_cell = None
                        solved_message = ""

if __name__ == "__main__":
    main()
